# Improved Hybrid Model: PhoBERT + TF-IDF (+ Enhancements)

## Cải tiến so với bản gốc:
1. **SMOTE** - Xử lý class imbalance cho Neutral class
2. **Regularization** - Giảm overfitting với C parameter tuning
3. **Late Fusion** - Kết hợp predictions thay vì concat features
4. **XGBoost** - Classifier mạnh hơn Logistic Regression
5. **Better preprocessing** - Trigrams, improved token filtering

## 1. Setup và Import Libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install transformers torch scikit-learn matplotlib seaborn xgboost imbalanced-learn underthesea

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from collections import Counter
import xgboost as xgb
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import random
import pickle
import re
import warnings
warnings.filterwarnings('ignore')

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

## 2. Configuration

In [ ]:
class Config:
    BASE_DIR = '/content/drive/MyDrive/Student-Feedback-Sentiment-Analysis'
    DATA_DIR = f'{BASE_DIR}/data/processed'
    RESULTS_DIR = f'{BASE_DIR}/results/TF_IDF_Hybrid_Improved'
    PHOBERT_MODEL_DIR = f'{BASE_DIR}/results/PhoBERT_Baseline/phobert_baseline_model.pt'
    MODEL_NAME = 'vinai/phobert-base'
    MAX_LENGTH = 256
    BATCH_SIZE = 32
    NUM_CLASSES = 3
    TFIDF_MAX_FEATURES = 3000
    TFIDF_NGRAM_RANGE = (1, 3)
    LABEL_MAP = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    SMOTE_SAMPLING_STRATEGY = {0: 5325, 1: 3000, 2: 5643}
    
config = Config()
os.makedirs(config.RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to: {config.RESULTS_DIR}')
print(f'Device: {config.DEVICE}')

In [ ]:
import pandas as pd
import json

comparison_data = []
for name, res in results.items():
    comparison_data.append({'Model': name, 'Accuracy': res['accuracy'], 'F1 (weighted)': res['f1']})
comparison_df = pd.DataFrame(comparison_data).sort_values('F1 (weighted)', ascending=False)

print('\n' + '='*60)
print('📊 MODEL COMPARISON')
print('='*60)
print(comparison_df.to_string(index=False))
comparison_df.to_csv(os.path.join(config.RESULTS_DIR, 'model_comparison.csv'), index=False)

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#4285F4', '#34A853', '#FBBC04']
axes[0].bar(comparison_df['Model'], comparison_df['Accuracy'], color=colors[:len(comparison_df)])
axes[0].set_title('Test Accuracy Comparison')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim(0.85, 1.0)
for i, v in enumerate(comparison_df['Accuracy']):
    axes[0].text(i, v + 0.002, f'{v:.4f}', ha='center', fontweight='bold')
axes[1].bar(comparison_df['Model'], comparison_df['F1 (weighted)'], color=colors[:len(comparison_df)])
axes[1].set_title('Test F1 Score Comparison')
axes[1].set_ylabel('F1 Score (weighted)')
axes[1].set_ylim(0.85, 1.0)
for i, v in enumerate(comparison_df['F1 (weighted)']):
    axes[1].text(i, v + 0.002, f'{v:.4f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(config.RESULTS_DIR, 'model_comparison.png'), dpi=150)
plt.show()

# Save best model and summary
best_model_name = comparison_df.iloc[0]['Model']
best_result = results[best_model_name]
with open(os.path.join(config.RESULTS_DIR, 'best_model.pkl'), 'wb') as f:
    pickle.dump(best_result['model'], f)

summary = {
    'Best Model': best_model_name,
    'Accuracy': best_result['accuracy'],
    'F1 (weighted)': best_result['f1'],
    'Improvements': [
        'SMOTE for class imbalance (Neutral: 458→3000)',
        'Reduced TF-IDF features (5000→3000)',
        'Stronger regularization (C: 1.0→0.1)',
        'Added trigrams (1,2)→(1,3)',
        'XGBoost classifier',
        'Late fusion approach'
    ]
}
with open(os.path.join(config.RESULTS_DIR, 'summary.json'), 'w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print(f'\n🏆 BEST MODEL: {best_model_name}')
print(f'   Accuracy: {best_result["accuracy"]:.4f}')
print(f'   F1 (weighted): {best_result["f1"]:.4f}')
print(f'\n✅ All results saved to: {config.RESULTS_DIR}')

## 8. Model Comparison & Save Results

In [ ]:
print('\n🔄 Training Late Fusion Model...')
scaler_phobert = StandardScaler()
train_phobert_scaled = scaler_phobert.fit_transform(train_phobert_emb)
test_phobert_scaled = scaler_phobert.transform(test_phobert_emb)
scaler_tfidf = StandardScaler()
train_tfidf_scaled = scaler_tfidf.fit_transform(train_tfidf.toarray())
test_tfidf_scaled = scaler_tfidf.transform(test_tfidf.toarray())

train_phobert_resampled, train_labels_res = smote.fit_resample(train_phobert_scaled, train_labels)
train_tfidf_resampled, _ = smote.fit_resample(train_tfidf_scaled, train_labels)

phobert_lr = LogisticRegression(multi_class='multinomial', max_iter=1000, random_state=42, n_jobs=-1, C=0.1)
phobert_lr.fit(train_phobert_resampled, train_labels_res)
phobert_probs = phobert_lr.predict_proba(test_phobert_scaled)
tfidf_lr = LogisticRegression(multi_class='multinomial', max_iter=1000, random_state=42, n_jobs=-1, C=0.1)
tfidf_lr.fit(train_tfidf_resampled, train_labels_res)
tfidf_probs = tfidf_lr.predict_proba(test_tfidf_scaled)

weights = {'phobert': 0.7, 'tfidf': 0.3}
fused_probs = weights['phobert'] * phobert_probs + weights['tfidf'] * tfidf_probs
fused_preds = np.argmax(fused_probs, axis=1)
fused_acc = accuracy_score(test_labels, fused_preds)
fused_f1 = f1_score(test_labels, fused_preds, average='weighted')
results['Late Fusion'] = {'model': {'phobert': phobert_lr, 'tfidf': tfidf_lr, 'weights': weights}, 'preds': fused_preds, 'accuracy': fused_acc, 'f1': fused_f1}
print(f'   Fusion weights: PhoBERT={weights["phobert"]}, TF-IDF={weights["tfidf"]}')
print(f'   Accuracy: {fused_acc:.4f}, F1: {fused_f1:.4f}')

In [ ]:
results = {}

print('🚂 Training Logistic Regression (C=0.1 for stronger regularization)...')
lr_model = LogisticRegression(
    multi_class='multinomial', max_iter=2000, random_state=42, n_jobs=-1, C=0.1, penalty='l2'
)
lr_model.fit(train_combined_scaled, train_labels_resampled)
lr_preds = lr_model.predict(test_combined_scaled)
lr_acc = accuracy_score(test_labels, lr_preds)
lr_f1 = f1_score(test_labels, lr_preds, average='weighted')
results['Logistic Regression'] = {'model': lr_model, 'preds': lr_preds, 'accuracy': lr_acc, 'f1': lr_f1}
print(f'   Accuracy: {lr_acc:.4f}, F1: {lr_f1:.4f}')

print('\n🚂 Training XGBoost...')
xgb_model = xgb.XGBClassifier(
    n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8,
    colsample_bytree=0.8, random_state=42, n_jobs=-1, eval_metric='mlogloss',
    use_label_encoder=False, reg_alpha=0.1, reg_lambda=1.0, min_child_weight=3
)
xgb_model.fit(train_combined_scaled, train_labels_resampled)
xgb_preds = xgb_model.predict(test_combined_scaled)
xgb_acc = accuracy_score(test_labels, xgb_preds)
xgb_f1 = f1_score(test_labels, xgb_preds, average='weighted')
results['XGBoost'] = {'model': xgb_model, 'preds': xgb_preds, 'accuracy': xgb_acc, 'f1': xgb_f1}
print(f'   Accuracy: {xgb_acc:.4f}, F1: {xgb_f1:.4f}')

## 7. Train Models

In [ ]:
print('='*60)
print('🔧 APPLYING SMOTE - FIX CLASS IMBALANCE')
print('='*60)
print('\nBefore SMOTE:')
print(Counter(train_labels))

smote = SMOTE(
    sampling_strategy=config.SMOTE_SAMPLING_STRATEGY,
    random_state=42,
    k_neighbors=5
)

train_combined_resampled, train_labels_resampled = smote.fit_resample(train_combined, train_labels)
print('\nAfter SMOTE:')
print(Counter(train_labels_resampled))
print(f'\nOriginal shape: {train_combined.shape}')
print(f'Resampled shape: {train_combined_resampled.shape}')
print(f'✅ SMOTE applied! Neutral class increased from 458 → 3000')

print('\n⚙️ Scaling features...')
scaler = StandardScaler()
train_combined_scaled = scaler.fit_transform(train_combined_resampled)
val_combined_scaled = scaler.transform(val_combined)
test_combined_scaled = scaler.transform(test_combined)
with open(os.path.join(config.RESULTS_DIR, 'feature_scaler.pkl'), 'wb') as f:
    pickle.dump(scaler, f)
print('✅ Features scaled!')

## 6. SMOTE - Fix Class Imbalance

In [ ]:
from tqdm import tqdm

def extract_phobert_embeddings(model, texts, tokenizer, device, batch_size=32):
    model.eval()
    all_embeddings = []
    dataloader = DataLoader(
        SentimentDataset(texts, [0]*len(texts), tokenizer, config.MAX_LENGTH),
        batch_size=batch_size, shuffle=False
    )
    with torch.no_grad():
        for batch in tqdm(dataloader, desc='Extracting embeddings'):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            embeddings = model.extract_embeddings(input_ids, attention_mask)
            all_embeddings.append(embeddings.cpu().numpy())
    return np.vstack(all_embeddings)

print('🔄 Extracting PhoBERT embeddings...')
train_phobert_emb = extract_phobert_embeddings(phobert_model, train_texts, tokenizer, config.DEVICE, config.BATCH_SIZE)
val_phobert_emb = extract_phobert_embeddings(phobert_model, val_texts, tokenizer, config.DEVICE, config.BATCH_SIZE)
test_phobert_emb = extract_phobert_embeddings(phobert_model, test_texts, tokenizer, config.DEVICE, config.BATCH_SIZE)
print(f'   Train: {train_phobert_emb.shape}, Val: {val_phobert_emb.shape}, Test: {test_phobert_emb.shape}')
print('✅ PhoBERT embeddings extracted!')

def combine_features(phobert_emb, tfidf_features):
    tfidf_dense = tfidf_features.toarray()
    combined = np.hstack([phobert_emb, tfidf_dense])
    return combined

print('🔗 Combining PhoBERT + TF-IDF features...')
train_combined = combine_features(train_phobert_emb, train_tfidf)
val_combined = combine_features(val_phobert_emb, val_tfidf)
test_combined = combine_features(test_phobert_emb, test_tfidf)
print(f'   Train combined shape: {train_combined.shape}')
print(f'   Val combined shape: {val_combined.shape}')
print(f'   Test combined shape: {test_combined.shape}')

In [ ]:
def preprocess_vietnamese(text):
    text = text.lower()
    text = re.sub(r'[^\w\sàáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]', ' ', text)
    text = ' '.join(text.split())
    return text

train_texts_processed = [preprocess_vietnamese(t) for t in train_texts]
val_texts_processed = [preprocess_vietnamese(t) for t in val_texts]
test_texts_processed = [preprocess_vietnamese(t) for t in test_texts]

print('📝 Creating improved TF-IDF vectorizer...')
tfidf_vectorizer = TfidfVectorizer(
    max_features=config.TFIDF_MAX_FEATURES,
    ngram_range=config.TFIDF_NGRAM_RANGE,
    min_df=3,
    max_df=0.90,
    sublinear_tf=True
)

train_tfidf = tfidf_vectorizer.fit_transform(train_texts_processed)
val_tfidf = tfidf_vectorizer.transform(val_texts_processed)
test_tfidf = tfidf_vectorizer.transform(test_texts_processed)

print(f'   TF-IDF train shape: {train_tfidf.shape}')
print(f'   TF-IDF val shape: {val_tfidf.shape}')
print(f'   TF-IDF test shape: {test_tfidf.shape}')

with open(os.path.join(config.RESULTS_DIR, 'tfidf_vectorizer.pkl'), 'wb') as f:
    pickle.dump(tfidf_vectorizer, f)
print(f'✅ TF-IDF vectorizer saved!')

## 5. TF-IDF Feature Extraction (Improved)

In [ ]:
class SentimentDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        encoding = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_length,
            padding='max_length', truncation=True, return_tensors='pt'
        )
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(config.MODEL_NAME)
print(f'Tokenizer loaded: {config.MODEL_NAME}')

class PhoBERTClassifier(nn.Module):
    def __init__(self, model_name, num_classes, dropout=0.1):
        super(PhoBERTClassifier, self).__init__()
        self.phobert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.phobert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        pooled_output = self.dropout(pooled_output)
        logits = self.classifier(pooled_output)
        return logits, pooled_output

    def extract_embeddings(self, input_ids, attention_mask):
        with torch.no_grad():
            outputs = self.phobert(input_ids=input_ids, attention_mask=attention_mask)
            pooled_output = outputs.last_hidden_state[:, 0, :]
        return pooled_output

def load_model_safe(model, checkpoint_path, device):
    checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=False)
    if isinstance(checkpoint, dict):
        if 'state_dict' in checkpoint:
            state_dict = checkpoint['state_dict']
        elif 'model_state_dict' in checkpoint:
            state_dict = checkpoint['model_state_dict']
        else:
            state_dict = checkpoint
    else:
        state_dict = checkpoint
    new_state_dict = {}
    for k, v in state_dict.items():
        if k.startswith('module.'):
            new_state_dict[k[7:]] = v
        else:
            new_state_dict[k] = v
    model.load_state_dict(new_state_dict)
    return model

print('📥 Loading PhoBERT Baseline model...')
phobert_model = PhoBERTClassifier(model_name=config.MODEL_NAME, num_classes=config.NUM_CLASSES)
phobert_model = load_model_safe(phobert_model, config.PHOBERT_MODEL_DIR, config.DEVICE)
phobert_model = phobert_model.to(config.DEVICE)
for param in phobert_model.parameters():
    param.requires_grad = False
phobert_model.eval()
print('✅ PhoBERT model loaded and frozen!')

## 4. PhoBERT Model Definition

In [ ]:
print('Label distribution (BEFORE SMOTE):')
for split_name, labels in [('Train', train_labels), ('Val', val_labels), ('Test', test_labels)]:
    counter = Counter(labels)
    print(f'{split_name}:')
    for label, count in sorted(counter.items()):
        print(f'  {config.LABEL_MAP[label]}: {count} ({count/len(labels)*100:.1f}%)')

In [ ]:
def load_data(data_dir, split):
    split_dir = os.path.join(data_dir, split)
    with open(os.path.join(split_dir, 'sents.txt'), 'r', encoding='utf-8') as f:
        texts = [line.strip() for line in f.readlines()]
    with open(os.path.join(split_dir, 'sentiments.txt'), 'r', encoding='utf-8') as f:
        labels = [int(line.strip()) for line in f.readlines()]
    print(f'{split}: {len(texts)} samples')
    return texts, labels

train_texts, train_labels = load_data(config.DATA_DIR, 'train')
val_texts, val_labels = load_data(config.DATA_DIR, 'validation')
test_texts, test_labels = load_data(config.DATA_DIR, 'test')

## 3. Load Data